### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd

In [2]:
from unsloth import FastModel
import torch

#unsloth/Qwen2.5-7B
#Qwen/Qwen2.5-Coder-7B-Instruct
#Qwen/Qwen2.5-Coder-3B
#unsloth/Llama-3.2-3B-Instruct

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = True, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)
print(model.dtype)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-24 22:32:26 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.18: Fast Llama patching. Transformers: 4.47.1. vLLM: 0.8.2.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
torch.bfloat16


In [3]:
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
#                       "gate_proj", "up_proj", "down_proj",],
#     lora_alpha = 16,
#     lora_dropout = 0, # Supports any, but = 0 is optimized
#     bias = "none",    # Supports any, but = "none" is optimized
#     # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
#     use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
#     random_state = 3407,
#     use_rslora = False,  # We support rank stabilized LoRA
#     loftq_config = None, # And LoftQ
# )

In [4]:
df=pd.read_csv('./drawing-with-llms/svg_score_extend_train1.csv')
df=df[['description','gpt_svg_1']]
print(df.shape)

(1975, 2)


In [5]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    topics = examples["description"]  # Using 'topic' as instruction
    svgs = examples["gpt_svg_1"]  # Using 'svg_code' as output
    texts = []
    
    for topic, svg_code in zip(topics, svgs):
        # No additional input is needed, so we pass an empty string
        text = alpaca_prompt.format(f"Generate an SVG image for the topic:",topic, svg_code) + EOS_TOKEN
        texts.append(text)
    
    return { "text": texts }

from datasets import Dataset
import pandas as pd
# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Apply formatting function
dataset = dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/1975 [00:00<?, ? examples/s]

In [6]:
# Check dataset sample output
print(dataset["text"][0])

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate an SVG image for the topic:

### Input:
'Golden wheat fields under a setting sun',

### Response:
<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">
  <!-- Background for the sky -->
  <rect x="0" y="0" width="200" height="100" fill="orange" opacity="0.7"/>
  
  <!-- Sun -->
  <circle cx="100" cy="50" r="30" fill="yellow" opacity="0.8"/>
  
  <!-- Wheat fields -->
  <rect x="0" y="100" width="200" height="100" fill="goldenrod"/>
  
  <!-- Wheat stalks -->
  <g stroke="saddlebrown" stroke-width="2">
    <line x1="30" y1="100" x2="30" y2="150"/>
    <line x1="50" y1="100" x2="50" y2="150"/>
    <line x1="70" y1="100" x2="70" y2="150"/>
    <line x1="90" y1="100" x2="90" y2="150"/>
    <line x1="110" y1="100" x2="110" y2="150"/>
    <line x1="130" y1="100" x2="130" y2="1

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, 
        max_steps = 1000,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit", # "adamw_torch" better for fp16
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 123,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1975 [00:00<?, ? examples/s]

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,975 | Num Epochs = 5 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,235,814,400/1,235,814,400 (100.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.616500
10,1.895800
15,1.075200
20,0.853900
25,0.732700
30,0.744700
35,0.694400
40,0.685500
45,0.562100
50,0.633400


In [9]:
#This ONLY saves the LoRA adapters, and not the full model.
model.save_pretrained("./lora/lora_model_3b_v3") # Local saving
tokenizer.save_pretrained("./lora/lora_model_3b_v3")

('./lora/lora_model_3b_v3/tokenizer_config.json',
 './lora/lora_model_3b_v3/special_tokens_map.json',
 './lora/lora_model_3b_v3/tokenizer.json')

In [10]:
# import gc
# del model
# gc.collect()
# torch.cuda.empty_cache()

In [11]:
# import psutil

# def kill_large_python_processes():
#     # Loop over all running processes
#     for proc in psutil.process_iter(['pid', 'name', 'memory_info', 'exe']):
#         try:
#             # Check if the process is Python and type is 'C' (for computation)
#             if 'python' in proc.info['name'].lower():
#                 # Check if memory usage is greater than 2048 MB
#                 memory_usage_mb = proc.info['memory_info'].rss / (1024 * 1024)  # Convert bytes to MB
#                 if memory_usage_mb > 2048:
#                     print(f"Killing Python process with PID {proc.info['pid']} using {memory_usage_mb} MB memory")
#                     proc.kill()  # Kill the process
#         except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
#             pass  # Handle processes that might disappear during iteration

# #kill_large_python_processes()

In [12]:
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
# model_name = "./lora/lora_model_3b_v2", # YOUR MODEL YOU USED FOR TRAINING
# max_seq_length = 2048,
# dtype = (None),
# load_in_4bit = False,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference


In [13]:
# # alpaca_prompt = You MUST copy from above!
# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# {}"""

# inputs = tokenizer(
# [
#     alpaca_prompt.format(
#         "Please write a SVG code fo rthe given topic?", # instruction
#         "Golden sun rising in the east", # input
#         "", # output - leave this blank for generation!
#     )
# ], return_tensors = "pt").to("cuda")

# outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
# tokenizer.batch_decode(outputs)

In [14]:
# #save merged 16bit
# model.save_pretrained_merged("./lora/lora_16bit_merged_3b_v3", tokenizer, save_method = "merged_16bit",)